# AirGuard — Feature Engineering Pipeline
Demonstrates construction of autoregressive lags, rolling statistics, diurnal cyclical encodings, and atmospheric dispersion indices without data leakage.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.database import fetch_historical_records
from src.preprocessing import clean_air_quality_dataframe
from src.feature_engineering import build_feature_pipeline, time_series_split

print('Feature engineering modules loaded.')

## 1. Load Clean Telemetry Data

In [ ]:
df_raw = fetch_historical_records(city='Chandigarh')
df_clean = clean_air_quality_dataframe(df_raw)
print(f'Loaded {len(df_clean)} cleaned records.')

## 2. Execute Feature Pipeline

In [ ]:
featured_df, feature_cols, target_cols = build_feature_pipeline(df_clean, horizons=[24, 72])
print(f'Constructed {len(feature_cols)} predictor features.')
print(f'Target horizons: {target_cols}')
featured_df[['timestamp', 'pm2_5', 'pm2_5_lag_1h', 'pm2_5_lag_24h', 'pm2_5_roll_mean_6h', 'hour_sin', 'hour_cos', 'wind_stagnation_index']].head(10)

## 3. Autoregressive Lag Correlation Analysis

In [ ]:
lag_cols = [c for c in feature_cols if 'lag' in c and 'pm2_5' in c]
corr_lags = featured_df[['pm2_5'] + lag_cols].corr()['pm2_5'].drop('pm2_5')

plt.figure(figsize=(10, 4))
corr_lags.plot(kind='bar', color='teal')
plt.title('Correlation of PM2.5 with Historical Autoregressive Lags')
plt.ylabel('Pearson Correlation Coefficient')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Strict Chronological Train/Val/Test Split

In [ ]:
train_df, val_df, test_df = time_series_split(featured_df.dropna(subset=feature_cols + ['target_lead_24h']))
print(f'Train samples: {len(train_df)}, Val samples: {len(val_df)}, Test samples: {len(test_df)}')
print(f'Chronological order preserved: {train_df.index.max() < val_df.index.min() < test_df.index.min()}')